In [6]:
import requests
import base64
import json
import time
import argparse
from pathlib import Path
from PIL import Image
from io import BytesIO
from dotenv import load_dotenv 
import os

# --- Configuration ---
load_dotenv()

api_key = os.getenv("RUNPOD_API_KEY")
if not api_key:
    raise ValueError("API key not found. Please create a .env file and add RUNPOD_API_KEY=your_key")

endpoint_id = os.getenv("RUNPOD_ENDPOINT_ID", "a48mrbdsbzg35n")

def load_workflow(workflow_path):
    """Load workflow from JSON file."""
    try:
        with open(workflow_path, 'r') as f:
            workflow_data = json.load(f)
        print(f"✓ Loaded workflow from: {workflow_path}")
        return workflow_data
    except FileNotFoundError:
        raise FileNotFoundError(f"Workflow file not found: {workflow_path}")
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid JSON in workflow file: {e}")

def update_workflow_seed(workflow, seed=None):
    """Update the seed in the workflow if a RandomNoise or KSampler node exists."""
    if seed is None:
        seed = int(time.time() * 1000) % (2**32)
    
    # Try to find and update seed in common node types
    if "workflow" in workflow.get("input", {}):
        nodes = workflow["input"]["workflow"]
    else:
        nodes = workflow
    
    for node_id, node_data in nodes.items():
        if isinstance(node_data, dict):
            # RandomNoise node (Chroma workflow style)
            if node_data.get("class_type") == "RandomNoise":
                node_data["inputs"]["noise_seed"] = seed
                print(f"✓ Updated RandomNoise seed to: {seed}")
            # KSampler node (Flux workflow style)
            elif node_data.get("class_type") == "KSampler":
                if "widgets_values" in node_data and len(node_data["widgets_values"]) > 0:
                    node_data["widgets_values"][0] = seed
                    print(f"✓ Updated KSampler seed to: {seed}")
    
    return workflow

def wrap_workflow(workflow):
    """Wrap workflow in RunPod API format if needed."""
    # If already wrapped, return as-is
    if "input" in workflow and "workflow" in workflow["input"]:
        return workflow
    
    # Otherwise, wrap it
    return {
        "input": {
            "workflow": workflow
        }
    }

def send_job(workflow_data):
    """Send job to RunPod API."""
    run_url = f"https://api.runpod.ai/v2/{endpoint_id}/run"
    
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    print("Sending request to start the job...")
    response = requests.post(run_url, headers=headers, json=workflow_data)
    
    if response.status_code == 200:
        response_data = response.json()
        job_id = response_data.get('id')
        
        if not job_id:
            print("Error: API response did not include a job ID.")
            print("Full response:", json.dumps(response_data, indent=2))
            return None
        
        print(f"✓ Job successfully started with ID: {job_id}")
        return job_id
    else:
        print(f"✗ Error: Initial API request failed with status code {response.status_code}")
        print("Response:", response.text)
        return None

def poll_job_status(job_id, max_polls=120, poll_interval=5):
    """Poll for job status until completion or timeout."""
    status_url = f"https://api.runpod.ai/v2/{endpoint_id}/status/{job_id}"
    
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    poll_count = 0
    
    while poll_count < max_polls:
        print(f"Polling for job status... (attempt {poll_count + 1}/{max_polls})")
        status_response = requests.get(status_url, headers=headers)
        status_data = status_response.json()
        job_status = status_data.get('status')

        if job_status == 'COMPLETED':
            print("✓ Job completed successfully!")
            return status_data
        
        elif job_status in ['IN_QUEUE', 'IN_PROGRESS']:
            time.sleep(poll_interval)
            poll_count += 1
        
        else:
            print(f"✗ Job execution failed or was cancelled.")
            print("Final Status:", job_status)
            if 'output' in status_data:
                print("Output/Error:", status_data['output'])
            else:
                print("Full response:", json.dumps(status_data, indent=2))
            return None
    
    print("✗ Timeout: Job did not complete within the expected time.")
    return None

def save_output_images(status_data, output_dir="outputs"):
    """Save generated images from the job output."""
    Path(output_dir).mkdir(exist_ok=True)
    
    try:
        images = status_data['output']['images']
        saved_files = []
        
        for idx, output_image_data in enumerate(images):
            image_base64 = output_image_data['data'] 
            filename = output_image_data.get('filename', f'output_{idx}.png')
            
            image_bytes = base64.b64decode(image_base64)
            image = Image.open(BytesIO(image_bytes))
            
            output_path = Path(output_dir) / filename
            image.save(output_path)
            saved_files.append(str(output_path))
            print(f"✓ Image saved: {output_path}")
        
        return saved_files
    
    except (KeyError, IndexError, TypeError) as e:
        print(f"✗ Error: Could not parse image data from the final response: {e}")
        
        # Redact base64 data for debugging
        debug_data = json.loads(json.dumps(status_data))
        try:
            for img in debug_data['output']['images']:
                if 'data' in img:
                    original_length = len(img['data'])
                    img['data'] = f"<base64 data redacted, original length: {original_length}>"
        except Exception:
            pass 
        
        print("Full response (redacted):", json.dumps(debug_data, indent=2))
        return []

def main():
    parser = argparse.ArgumentParser(
        description='Run ComfyUI workflows on RunPod',
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog="""
Examples:
  python runpod_runner.py workflow.json
  python runpod_runner.py workflow.json --seed 12345
  python runpod_runner.py workflow.json --output my_outputs --timeout 600
        """
    )
    parser.add_argument('workflow', type=str, help='Path to workflow JSON file')
    parser.add_argument('--seed', type=int, default=None, help='Random seed (default: time-based)')
    parser.add_argument('--output', type=str, default='outputs', help='Output directory (default: outputs)')
    parser.add_argument('--timeout', type=int, default=600, help='Max wait time in seconds (default: 600)')
    parser.add_argument('--poll-interval', type=int, default=5, help='Polling interval in seconds (default: 5)')
    
    args = parser.parse_args()
    
    # Load and prepare workflow
    workflow = load_workflow(args.workflow)
    workflow = update_workflow_seed(workflow, args.seed)
    workflow = wrap_workflow(workflow)
    
    # Send job
    job_id = send_job(workflow)
    if not job_id:
        return
    
    # Poll for completion
    max_polls = args.timeout // args.poll_interval
    status_data = poll_job_status(job_id, max_polls=max_polls, poll_interval=args.poll_interval)
    
    if status_data:
        # Save output images
        saved_files = save_output_images(status_data, args.output)
        if saved_files:
            print(f"\n✓ Successfully generated {len(saved_files)} image(s)")
        else:
            print("\n✗ No images were generated")
    else:
        print("\n✗ Job failed or timed out")

if __name__ == "__main__":
    main()
    run_workflow(
    workflow_path='flux_dev_full_text_to_image.json',
    seed=12345,
    output_dir='my_outputs',
    timeout=900,
    poll_interval=5
)
    

usage: ipykernel_launcher.py [-h] [--seed SEED] [--output OUTPUT]
                             [--timeout TIMEOUT]
                             [--poll-interval POLL_INTERVAL]
                             workflow
ipykernel_launcher.py: error: unrecognized arguments: -f


SystemExit: 2

In [5]:
# Cell 1: Import and setup (run once)
from runpod_runner import run_workflow

# Cell 2: Run a workflow - simple usage
run_workflow('chroma_workflow.json')

# Cell 3: Run with custom parameters
run_workflow(
    workflow_path='flux_dev_full_text_to_image.json',
    seed=12345,
    output_dir='my_outputs',
    timeout=900,
    poll_interval=5
)

# Cell 4: Run multiple workflows
workflows = ['workflow1.json', 'workflow2.json', 'workflow3.json']
for wf in workflows:
    print(f"\n--- Running {wf} ---")
    run_workflow(wf)

ModuleNotFoundError: No module named 'runpod_runner'